# HALO Phase 2 — quantized generation

This notebook runs one model at **three precisions** — fp16, 8-bit
(LLM.int8) and 4-bit (NF4) — and records, for every question:

- the answer text,
- the log-probability of every generated token (Phase 3's confidence signal),
- the retrieved chunks the model saw (Phase 3's grounding signal),
- latency and peak GPU memory.

**Requirements:** a CUDA GPU (Colab: *Runtime → Change runtime type → T4*).
`bitsandbytes` has no CPU or MPS backend, so quantized loading fails
otherwise.

**Fairness:** the three `configs/*.yaml` files differ only in `precision`
and `output` — a unit test pins that. Retrieval is computed once and shared,
so the only thing that changes between runs is how the weights are stored.

**Budget:** 100 questions × 3 precisions, greedy decoding — roughly 45–60
minutes on a free T4. Lower `questions` in the YAML files for a faster
smoke run.

Nothing is pre-filled: every number below is computed from this run.

In [ ]:
import importlib.util

ON_COLAB = importlib.util.find_spec("google.colab") is not None
if ON_COLAB:
    !git clone --depth 1 https://github.com/reyyishreyas/halo.git
    %cd halo
    !pip install -q -e ".[quantize]"
else:
    # local run: assume the notebook's parent repo is already installed
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path.cwd().parent))
    print("local mode: using the repo this notebook lives in")

In [ ]:
import torch

import halo

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print("GPU: Apple MPS (fp16 only; int8/nf4 need CUDA + bitsandbytes)")
else:
    raise AssertionError(
        "No GPU found. On Colab: Runtime -> Change runtime type -> T4 GPU."
    )
print("halo", halo.__version__, "| torch", torch.__version__)

## 1. The three run configs

Each file fixes the whole run: model, top-k, token budget, question
count, seed. Only `precision` and `output` differ — `tests/test_generate.py`
fails if that ever drifts.

**Smoke vs full:** the next cell has a `SMOKE_QUESTIONS` toggle. Set it
to a small number (e.g. `3`) for a quick end-to-end check — a few
minutes. Set it to `None` for the real 100-question run that produces
the numbers for the report.


In [ ]:
import dataclasses

from halo.generate import load_settings

# Set to None for the full 100-question run; a small number (e.g. 3)
# gives a fast end-to-end smoke test without editing the YAML files.
SMOKE_QUESTIONS = 3

configs = [load_settings(f"configs/{p}.yaml") for p in ("fp16", "int8", "nf4")]
if SMOKE_QUESTIONS is not None:
    configs = [dataclasses.replace(s, questions=SMOKE_QUESTIONS) for s in configs]
    print(f"SMOKE MODE: {SMOKE_QUESTIONS} questions per precision")
else:
    print("FULL RUN: using each config's own question count")
print()
for settings in configs:
    print(settings)


## 2. Retrieval — computed once, shared by all three runs

Held-out test split (the seeded half of HotpotQA's validation set). The
retrieved hits are cached here so quantization cannot change what the
model saw.

In [ ]:
from sentence_transformers import SentenceTransformer

from halo.chunk import chunk_paragraphs
from halo.data import make_splits
from halo.index import DEFAULT_EMBEDDER, build_index, load_tokenizer

n_questions = max(settings.questions for settings in configs)
records = make_splits()["test"][:n_questions]
codec = load_tokenizer()
encoder = SentenceTransformer(DEFAULT_EMBEDDER, device=DEVICE)

retrieval = []
for record in records:
    chunks = chunk_paragraphs(record.paragraphs, tokenizer=codec)
    if not chunks:
        print(f"skipping {record.id}: no chunks")
        continue
    index = build_index(chunks, encoder=encoder)
    retrieval.append((record, index.search(record.question, configs[0].top_k)))

print(f"cached retrieval for {len(retrieval)} questions")


## 3. Generate at each precision

One precision at a time so the 4-bit load never shares memory with
fp16. Each run is written to its own JSONL file as soon as it finishes,
so a dropped session loses at most the run in flight.

In [ ]:
import gc

import torch

from halo.generate import (
    answer_question,
    load_generator,
    summarize,
    write_jsonl,
)

runnable = [
    s for s in configs
    if s.precision == "fp16" or DEVICE == "cuda"
]
for skipped in {s.precision for s in configs} - {s.precision for s in runnable}:
    print(f"skipping {skipped}: needs CUDA + bitsandbytes (run on Colab T4)")

for settings in runnable:
    print(f"=== {settings.precision} ===", flush=True)
    model, tokenizer = load_generator(settings, device=DEVICE)
    run = [
        answer_question(model, tokenizer, record, hits, settings)
        for record, hits in retrieval
    ]
    write_jsonl(settings.output, run)
    print(summarize(run), flush=True)
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


## 4. Comparison table

Built from the JSONL files just written — the same numbers a fresh
session would recompute.

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for settings in configs:
    if not Path(settings.output).exists():
        print(f"no {settings.output} on this machine - run that precision elsewhere")
        continue
    with open(settings.output) as handle:
        run = [json.loads(line) for line in handle]
    rows.append({"precision": settings.precision, **summarize(run)})

table = pd.DataFrame(rows).set_index("precision")
table


Same question, three precisions: the answer, its mean token
log-probability, and latency.

In [ ]:
from pathlib import Path

for settings in configs:
    if not Path(settings.output).exists():
        continue
    with open(settings.output) as handle:
        first = json.loads(handle.readline())
    mean_lp = sum(first["token_logprobs"]) / len(first["token_logprobs"])
    print(f"[{settings.precision}] {first['question']}")
    print(
        f"  mean token log-prob {mean_lp:.3f} | "
        f"{first['latency_s']:.2f}s | peak {first['peak_gpu_mb']} MB"
    )
    print(f"  {first['answer'][:280]}\n")


## 5. Keep the outputs

The three JSONL files under `artifacts/generations/` are Phase 3's
input: sentence spans are cut from `token_ids` / `token_logprobs`, and
each span's grounding score embeds it against the `retrieved` chunks
stored in the same record. Download them (or grab them from the file
browser on the left) before the runtime disconnects.

In [ ]:
from google.colab import files

for settings in configs:
    files.download(settings.output)